## 加载数据

数据已经预处理过了

In [1]:
import pandas as pd

# 加载数据
data = pd.read_csv('../../data-hh/my/hh_result/result_all.csv', dtype={'aircraft': str})

# 查看前几行数据，确保加载成功
print(data.head())

         flt_no bd_type    cap aircraft  legs  leg_no  duration  pax  \
0  KgJrsp7Jd78=      窄体  132.0      319     1       1      1.07   25   
1  P9IRwar34h0=      窄体  189.0      321     1       1      1.38  151   
2  mJitm0UDfM4=      窄体  132.0      319     1       1      1.57   38   
3  jXr97M1wpn4=      窄体  132.0      319     1       1      1.58  109   
4  izjfHOxAho4=      窄体  132.0      319     1       1      1.80  124   

              a             b  ...  year  month  day  weekday  hour  minute  \
0  KNqX4/Q5Noc=  HexFWXqbb8I=  ...  2023     10    1        6    12      15   
1  AKQNtuL5r6Q=  Mv6HkAiSLUk=  ...  2023     10    1        6    20      10   
2  n465JzB8Rrw=  N4hmDZN/CJQ=  ...  2023     10    1        6    16      45   
3  N4hmDZN/CJQ=  n465JzB8Rrw=  ...  2023     10    1        6    19      40   
4  5t+HPO9Mu/w=  X5e5r3CS4OA=  ...  2023     10    1        6    13       5   

   second          from            to   unit_price  
0       0  KNqX4/Q5Noc=  HexFWXqbb8I=  

## 编码分类变量

In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder
import os

from joblib import dump

# 1. 获取所有唯一城市集合
# all_cities = pd.unique(data[['a', 'b', 'c', 'from', 'to']].values.ravel())

# 2. 为每列生成独热编码规则并对数据进行编码
encoders = {}
encoded_features = []

for col in ['a', 'b', 'c', 'from', 'to','flt_no', 'bd_type', 'aircraft']:
    # 生成独热编码规则
    encoder = pd.get_dummies(data[col], prefix=col).columns.tolist()
    encoders[col] = encoder
    
    # 对当前列进行独热编码
    encoded_col = pd.get_dummies(data[col], prefix=col)
    encoded_features.append(encoded_col)

# 合并独热编码后的特征到原始数据
data = pd.concat([data] + encoded_features, axis=1)

# 删除原始的 'a', 'b', 'c', 'from', 'to' 列
data = data.drop(columns=['a', 'b', 'c', 'from', 'to','flt_no', 'bd_type', 'aircraft'])



print("\n编码后的 DataFrame:")
print(data)


In [5]:
# 查看处理后的数据
print(data.head())

         flt_no bd_type    cap aircraft  legs  leg_no  duration  pax  year  \
0  KgJrsp7Jd78=      窄体  132.0      319     1       1      1.07   25  2023   
1  P9IRwar34h0=      窄体  189.0      321     1       1      1.38  151  2023   
2  mJitm0UDfM4=      窄体  132.0      319     1       1      1.57   38  2023   
3  jXr97M1wpn4=      窄体  132.0      319     1       1      1.58  109  2023   
4  izjfHOxAho4=      窄体  132.0      319     1       1      1.80  124  2023   

   month  ...  to_y+pJ91YLvhA=  to_yRnn9dxWrPQ=  to_ycSPshXrSuw=  \
0     10  ...            False            False            False   
1     10  ...            False            False            False   
2     10  ...            False            False            False   
3     10  ...            False            False            False   
4     10  ...            False            False            False   

   to_yhlbDZphNw8=  to_yypkiQCX5lk=  to_z+SilYYwbDk=  to_z97p/aY3afQ=  \
0            False            False            Fa

## 特征和目标分离
我们要预测的是pax字段，其他字段作为特征。

In [6]:
# 目标列
y = data['pax']

# 特征列（去除目标列 'pax' 和可能的非特征列）
X = data.drop(columns=['pax'])

# 查看特征列数目
print("特征列数目:", X.shape[1])

特征列数目: 1223


## 训练XGBoost模型

In [7]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 假设 X 和 y 是你的特征和标签数据
# 第一轮划分：训练集 80%，临时集 20%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# 第二轮划分：临时集 50% 为验证集，50% 为测试集，保证验证集和测试集各占原始数据的 10%
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 输出每个数据集的大小
print(f'训练集大小: {X_train.shape[0]}')
print(f'验证集大小: {X_val.shape[0]}')
print(f'测试集大小: {X_test.shape[0]}')

# 创建XGBoost回归模型
model = xgb.XGBRegressor(
    objective='reg:squarederror', 
    learning_rate=0.01, 
    max_depth=6, 
    n_estimators=1000, 
    subsample=0.8, 
    colsample_bytree=0.7, 
    alpha=10,
    early_stopping_rounds=10,  # Enable early stopping
)

# 设置验证集以监控进展
eval_set = [(X_train, y_train), (X_val, y_val)]  # 第一个是训练集，第二个是验证集

# 训练模型，同时显示进展
model.set_params(eval_metric="rmse")  # 设置评估指标
model.fit(
    X_train, y_train, 
    eval_set=eval_set, 
    verbose=500  # 每轮显示进展
)

# 预测测试集
y_pred = model.predict(X_test)

# 评估模型
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error on Test Set: {mse}')

训练集大小: 4799380
验证集大小: 599922
测试集大小: 599923


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:flt_no: object, bd_type: object, aircraft: object

In [10]:
# 显示20条测试结果（真实值 vs 预测值）
test_results = list(zip(y_test.values[:100], y_pred[:100]))  # 真实值和预测值
print("\n20条测试结果（真实值 vs 预测值）:")
for i, (true_value, pred_value) in enumerate(test_results):
    print(f"第{i+1}条: 真实值={true_value}, 预测值={pred_value:.2f}")


20条测试结果（真实值 vs 预测值）:
第1条: 真实值=91, 预测值=74.31
第2条: 真实值=255, 预测值=210.37
第3条: 真实值=163, 预测值=155.47
第4条: 真实值=234, 预测值=226.37
第5条: 真实值=82, 预测值=109.77
第6条: 真实值=88, 预测值=85.42
第7条: 真实值=66, 预测值=77.96
第8条: 真实值=224, 预测值=199.65
第9条: 真实值=181, 预测值=168.84
第10条: 真实值=131, 预测值=123.80
第11条: 真实值=116, 预测值=146.16
第12条: 真实值=99, 预测值=130.73
第13条: 真实值=127, 预测值=108.44
第14条: 真实值=30, 预测值=52.80
第15条: 真实值=106, 预测值=104.31
第16条: 真实值=155, 预测值=139.73
第17条: 真实值=139, 预测值=141.28
第18条: 真实值=28, 预测值=49.02
第19条: 真实值=88, 预测值=123.66
第20条: 真实值=200, 预测值=189.83
第21条: 真实值=102, 预测值=129.84
第22条: 真实值=93, 预测值=95.13
第23条: 真实值=118, 预测值=146.33
第24条: 真实值=182, 预测值=173.15
第25条: 真实值=69, 预测值=109.74
第26条: 真实值=49, 预测值=62.73
第27条: 真实值=70, 预测值=71.00
第28条: 真实值=146, 预测值=130.57
第29条: 真实值=31, 预测值=37.95
第30条: 真实值=179, 预测值=159.64
第31条: 真实值=168, 预测值=161.10
第32条: 真实值=174, 预测值=162.15
第33条: 真实值=124, 预测值=115.82
第34条: 真实值=96, 预测值=101.11
第35条: 真实值=132, 预测值=135.22
第36条: 真实值=162, 预测值=143.78
第37条: 真实值=144, 预测值=136.33
第38条: 真实值=185, 预测值=177.84
第39条: 真实值=130, 预测值=104

似乎对于较小值预测存在误差

In [11]:
def calculate_smape(y_true, y_pred):
    """
    计算 Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))
    return smape

def calculate_mape(y_true, y_pred):
    """
    计算 Mean Absolute Percentage Error (MAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    return mape

In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 评估模型
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = calculate_mape(y_test, y_pred)
smape = calculate_smape(y_test, y_pred)

# 打印结果
print(f'Mean Squared Error (MSE): {mse:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}%')
print(f'Symmetric Mean Absolute Percentage Error (SMAPE): {smape:.4f}%')

Mean Squared Error (MSE): 393.1527
Root Mean Squared Error (RMSE): 19.8281
Mean Absolute Error (MAE): 15.1215
Mean Absolute Percentage Error (MAPE): 17.8068%
Symmetric Mean Absolute Percentage Error (SMAPE): 15.8580%


## 保存模型

In [13]:
model.save_model("../../data-hh/my/xgboost_model_200000.json")
print("模型已保存为 xgboost_model.json")

模型已保存为 xgboost_model.json


## 超参数设置

In [5]:
import re
import pandas as pd
import ace_tools as tools;

# Input data
data = """
[0] validation_0-rmse:45.28687 validation_1-rmse:45.30570
[500] validation_0-rmse:28.47916 validation_1-rmse:28.55396
[1000] validation_0-rmse:27.42265 validation_1-rmse:27.50307
[1500] validation_0-rmse:26.79855 validation_1-rmse:26.88436
[2000] validation_0-rmse:26.35383 validation_1-rmse:26.44519
"""

# Regular expression to extract data
pattern = r"\[(\d+)\]\s+validation_0-rmse:([\d.]+)\s+validation_1-rmse:([\d.]+)"
matches = re.findall(pattern, data)

# Convert to DataFrame
df = pd.DataFrame(matches, columns=["n_estimators", "validation_0_rmse", "validation_1_rmse"])
df["n_estimators"] = df["n_estimators"].astype(int)
df["validation_0_rmse"] = df["validation_0_rmse"].astype(float)
df["validation_1_rmse"] = df["validation_1_rmse"].astype(float)

# Display the result
tools.display_dataframe_to_user(name="Extracted RMSE Data", dataframe=df)

ModuleNotFoundError: No module named 'ace_tools'